# Model Prototyping

This notebook builds a content-based movie recommender. It prepares text features, converts them into TF-IDF vectors, recommends similar movies with cosine similarity, and saves the model artifacts.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import ast
import string

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nojan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\nojan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\nojan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Load Dataset

Read the movie metadata CSV from `data/raw`. The path check supports running the notebook from either the project root or the `notebooks` folder.

In [4]:
from pathlib import Path

data_path = Path('data/raw/movies_metadata.csv')
if not data_path.exists():
    data_path = Path('../data/raw/movies_metadata.csv')

if not data_path.exists():
    raise FileNotFoundError("Place movies_metadata.csv in the project's data/raw/ folder.")

df = pd.read_csv(data_path)
display(df.head())

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


## Prepare Movie Data

Remove duplicate rows, keep only the fields needed for recommendations, fill missing text values, and convert nested metadata strings into usable text.

In [5]:
df = df.drop_duplicates().reset_index(drop=True)

In [6]:
movie_df = df[['title', 'genres', 'overview', 'tagline', 'popularity', 'belongs_to_collection', 'vote_average']]

In [7]:
movie_df.dropna(inplace=True, axis=0, subset=['title'])

In [8]:
movie_df['overview'] = movie_df['overview'].fillna('')

In [9]:
movie_df['genres'] = movie_df['genres'].apply(lambda x: " ".join([i['name'] for i in ast.literal_eval(x)]))

In [10]:
movie_df['belongs_to_collection'] = movie_df['belongs_to_collection'].apply(
    lambda x: ast.literal_eval(x)['name'] if isinstance(x, str) else " "
)

In [11]:
movie_df['tagline'] = movie_df['tagline'].fillna('')

## Build Combined Tags

Combine genres, overview, tagline, and collection name into one `tags` column. This gives each movie a single text profile for content-based comparison.

In [12]:
movie_df['tags'] = movie_df['genres']+ " " + movie_df['overview']+ " " + movie_df['tagline']+ " " + movie_df['belongs_to_collection']

## Preprocess Text

Prepare the tag text for modeling by lowercasing it, replacing punctuation, tokenizing words, removing stopwords, and lemmatizing the remaining words.

In [13]:
translator = str.maketrans(string.punctuation, ' '*len(string.punctuation))
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

In [14]:
def preprocess_text(text):
  #Lower case
  text = text.lower()

  #Remove Punctuations
  text = text.translate(translator)

  #Tokenize
  tokens = word_tokenize(text)
  #Remove stop_words then lemmatize remaining words
  tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

  return " ".join(tokens)

In [15]:
movie_df['tags'] = movie_df['tags'].apply(preprocess_text)

In [16]:
movie_df.reset_index(drop = True, inplace = True)

## Build TF-IDF Features

Convert each movie's processed tag text into TF-IDF vectors. These vectors capture which words and short phrases are important for each movie compared with the rest of the dataset.

In [17]:
tfidf = TfidfVectorizer(stop_words = 'english', max_features=75000, ngram_range=(1,2))

In [18]:
tfidf_matrix = tfidf.fit_transform(movie_df['tags'])

In [19]:
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1679720 stored elements and shape (45447, 75000)>

## Create Title Index

Build a lookup table from lowercase movie titles to dataframe row indexes. This lets the recommender quickly find the selected movie before calculating similarity scores.

In [20]:
indices = pd.Series(movie_df.index, index = movie_df['title'].str.lower()).drop_duplicates()

In [21]:
indices

title
toy story                          0
jumanji                            1
grumpier old men                   2
waiting to exhale                  3
father of the bride part ii        4
                               ...  
subdue                         45442
century of birthing            45443
betrayal                       45444
satan triumphant               45445
queerama                       45446
Length: 45447, dtype: int64

## Create Recommendation Function

The function lowercases the requested title, finds that movie's row index, compares its TF-IDF vector against every other movie with cosine similarity, and returns the closest matches.

In [22]:
def recommend(title, n = 10):
  title = title.lower()
  if title not in indices:
    return ['Movie not found']

  idx = indices[title]

  if isinstance(idx, pd.Series):
    idx = idx.iloc[0]

  sim_score = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
  similar_idx = sim_score.argsort()[::-1][1:n+1]

  return movie_df['title'].iloc[similar_idx]

## Test the Recommender

Call the recommendation function with a known movie title to confirm that it returns similar movies based on the TF-IDF tag similarity.

In [23]:
recommend("Pirates of the Caribbean: The Curse of the Black Pearl")

11005          Pirates of the Caribbean: Dead Man's Chest
17119         Pirates of the Caribbean: On Stranger Tides
11824            Pirates of the Caribbean: At World's End
26198                                     The Golden Hawk
26548    Pirates of the Caribbean: Dead Men Tell No Tales
44626                                          God of War
30214                     Arabella, the Pirate's Daughter
12746                                      The Black Swan
22220                             The Boy and the Pirates
40161                                         Last Hijack
Name: title, dtype: str

## Save Model Artifacts

Save the trained TF-IDF vectorizer, sparse matrix, title index, and cleaned movie dataframe into the `models` folder so the recommender can be reused later without rebuilding everything from scratch.

In [24]:
import pickle

# 1. Save the precomputed TF-IDF matrix
pickle.dump(tfidf_matrix, open('../models/tfidf_matrix.pkl', 'wb'))

# 2. Save the title-to-index mapping
pickle.dump(indices, open('../models/indices.pkl', 'wb'))

# 3. Save the clean dataframe to retrieve movie details later
movie_df.to_pickle('../models/movie_df.pkl')

# 4. Save the fitted vectorizer (useful for custom text queries later)
pickle.dump(tfidf, open('../models/tfidf.pkl', 'wb'))

print("Artifacts successfully saved for deployment!")

Artifacts successfully saved for deployment!
